# Layer placement and needling operations

These recipe commands approximate manufacturing by applying temporary
device-resident targets. They must be separated by explicit relaxation
operations. Releasing a target lets the displaced fibers and their
contacts settle mechanically.

In [ ]:
import tangle

recipe = tangle.Recipe(
    tangle.Cell([1e-3, 1e-3, 3e-3], periodic=[True, True, False]),
    # 0=x, 1=y, 2=z; z is the layer and needle-motion direction.
    layer_axis=2,
)

## Layer commands

- `move_layers(spacing_scale, stiffness, max_translation)` scales all
  current layer-center spacings.
- `place_layer_above(layer, gap, ...)` brings one layer to a surface gap
  above the active stack.
- `release_layer_targets()` removes those temporary clamps.
- `fit_cell_to_active_fibers(axes, padding)` removes empty domain space.

In [ ]:
# move_layers scales current layer-center spacing; it does not teleport
# fibers or bypass contact relaxation.
recipe.move_layers(0.8, stiffness=0.5, max_translation=5e-6)
recipe.relax_for(500)
recipe.place_layer_above(2, gap=2e-6, stiffness=0.5, max_translation=5e-6)
recipe.relax_until_targets_reached(0.1e-6, 2_000)
# Releasing temporary clamps lets later stages move the stack freely.
recipe.release_layer_targets()
recipe.fit_cell_to_active_fibers(
    axes=[False, False, True], padding=25e-6
)

## Needle commands

`needle_layer_circular` selects one internal vertex from each eligible
fiber intersecting a circular footprint. `needle_layer_random` samples
eligible fibers by fraction. Both accept depth, optional minimum fiber
diameter, target stiffness, absolute maximum motion, and a motion cap
relative to the selected fiber diameter. Needle locations can be made
random by sampling `center` in Python before adding the operation.

In [ ]:
# Circular needling selects one internal vertex per eligible fiber in
# the x/y footprint because z is the configured layer axis.
recipe.needle_layer_circular(
    layer=2, center=[0.45e-3, 0.55e-3], diameter=100e-6,
    depth=0.6e-3, minimum_fiber_diameter=15e-6,
    stiffness=0.75, max_translation=5e-6,
    maximum_translation_over_fiber_diameter=0.5,
)
recipe.relax_until_targets_reached(0.1e-6, 5_000)
# Hold the target through relaxation, then release it before continuing.
recipe.release_needles()

# Random needling samples fibers reproducibly from seed rather than by
# spatial footprint.
recipe.needle_layer_random(
    layer=3, fraction=0.15, depth=0.6e-3, seed=2026,
    minimum_fiber_diameter=15e-6, stiffness=0.75,
    max_translation=5e-6,
    maximum_translation_over_fiber_diameter=0.5,
)
recipe.release_needles()
print(*recipe.operations(), sep="\n")